In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RadarComparison"
# dataType = "RadarComparison_Interpolation"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

# Region = "Hawaii"; Case = "WET"; spinup_hours = "12"; spinup_hours="-16"
# Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#Importing ERA5 Data Loading Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","ERA5_Data"))
from CLASSES_ERA5DataLoading import ERA5DataLoading_Class,ERA5DataLoading_Class_gdex

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Plotting import ContourPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=20
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData_NSSL.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetLoopElements(start_job,end_job):
    loop_elements = np.arange(ModelData_NSSL.Ntime)[start_job:end_job].tolist()
    return loop_elements
loop_elements = GetLoopElements(start_job,end_job)

In [ ]:
####################################
#DATA LOADING

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
# # import pyart
# # # https://arm-doe.github.io/pyart/API/generated/pyart.retrieve.create_cfad.html
# pyart.retrieve.create_cfad??

# # ==>

# Signature:
# pyart.retrieve.create_cfad(
#     radar,
#     field_bins,
#     altitude_bins,
#     field='reflectivity',
#     field_mask=None,
#     min_frac_thres=0.1,
# )
# Source:   
# def create_cfad(
#     radar,
#     field_bins,
#     altitude_bins,
#     field="reflectivity",
#     field_mask=None,
#     min_frac_thres=0.1,
# ):
#     """
#     This function returns a Contoured Frequency by Altitude Diagram (CFAD; Yuter et al. 1995), a 2-dimensional
#     histogram that is normalized by the number of points at each altitude. Altitude bins are masked where the counts
#     are less than a minimum fraction of the largest number of counts for any altitude row.

#     Author: Laura Tomkins (lauramtomkins@gmail.com)

#     Parameters
#     ----------
#     radar : Radar
#         Radar object used. Can be Radar or Grid object.
#     field_bins : list
#         List of bin edges for field values to use for CFAD creation.
#     altitude_bins : list
#         List of bin edges for height values to use for CFAD creation.
#     field : str
#         Field name to use to look up reflectivity data. In the
#         radar object. Default field name is 'reflectivity'.
#     field_mask : array
#         An array the same size as the field array used to mask values.
#     min_frac_thres : float, optional
#         Fraction of values to remove in CFAD normalization (default 0.1). If an altitude row has a total count that
#         is less than min_frac_thres of the largest number of total counts for any altitude row, the bins in that
#         altitude row are masked.

#     Returns
#     -------
#     freq_norm : array
#         Array of normalized frequency.
#     height_edges : array
#         Array of bin edges for height data.
#     field_edges : array of x coordinates
#         Array of bin edges for field data.

#     References
#     ----------
#     Yuter, S. E., and R. A. Houze, 1995: Three-Dimensional Kinematic and
#     Microphysical Evolution of Florida Cumulonimbus. Part II: Frequency Distributions
#     of Vertical Velocity, Reflectivity, and Differential Reflectivity. Mon. Wea. Rev.
#     123, 1941-1963. https://doi.org/10.1175/1520-0493(1995)123%3C1941:TDKAME%3E2.0.CO;2


#     """

#     # get field data
#     field_data = radar.fields[field]["data"][:]

#     # get altitude data
#     # first try to get altitude data from a radar object
#     try:
#         altitude_data = radar.gate_z["data"]
#     # if it fails, try to get altitude data from a grid object
#     except:
#         try:
#             altitude_data = radar.point_z["data"]
#         except:
#             print("No altitude data found")
#             raise

#     # option to mask data if a mask is given
#     if field_mask is not None:
#         field_data = np.ma.masked_where(field_mask, field_data)
#         altitude_data = np.ma.masked_where(field_data.mask, altitude_data)
#     else:
#         if isinstance(field_data, np.ma.MaskedArray):
#             mask = field_data.mask
#             altitude_data = np.ma.masked_where(mask, altitude_data)

#     # get raw bin counts
#     freq, height_edges, field_edges = np.histogram2d(
#         altitude_data.compressed(),
#         field_data.compressed(),
#         bins=[altitude_bins, field_bins],
#     )

#     # sum counts over y axis (height)
#     freq_sum = np.sum(freq, axis=1)
#     # get threshold for normalizing
#     point_thres = min_frac_thres * np.max(freq_sum)
#     # repeat to create array same size as freq
#     freq_sum_rep = np.repeat(freq_sum[..., np.newaxis], freq.shape[1], axis=1)
#     # normalize
#     freq_norm = freq / freq_sum_rep
#     # mask data where there is not enough points
#     freq_norm = np.ma.masked_where(freq_sum_rep < point_thres, freq_norm)

#     return freq_norm, height_edges, field_edges
# File:      ~/.local/lib/python3.10/site-packages/pyart/retrieve/cfad.py
# Type:      function

In [ ]:
def create_cfad_gridded(
    radar_gridded,
    altitude_levels,
    field_bins,
    altitude_bins,
    field_mask=None,
    min_frac_thres=0.1,
    normalize=True
):
    """
    Gridded-data version of PyART's create_cfad().
    """

    # -----------------------------
    # PyART line: field_data = radar.fields[field]["data"][:]
    # Replace with gridded data
    # -----------------------------
    field_data = np.asarray(radar_gridded)

    # -----------------------------
    # PyART line: altitude_data = radar.gate_z[...] or radar.point_z
    # Replace with broadcasted altitude array
    # -----------------------------
    nz, ny, nx = field_data.shape
    altitude_data = np.repeat(
        altitude_levels[:, None, None], ny, axis=1
    )
    altitude_data = np.repeat(altitude_data, nx, axis=2)

    # -----------------------------
    # Apply mask (PyART logic preserved)
    # -----------------------------
    if field_mask is not None:
        field_data = np.ma.masked_where(field_mask, field_data)
        altitude_data = np.ma.masked_where(field_data.mask, altitude_data)
    else:
        # If input field already masked
        if isinstance(field_data, np.ma.MaskedArray):
            mask = field_data.mask
            altitude_data = np.ma.masked_where(mask, altitude_data)
        else:
            # Mask invalid numbers (NaNs)
            field_data = np.ma.masked_invalid(field_data)
            altitude_data = np.ma.masked_where(field_data.mask, altitude_data)

    # -----------------------------
    # PyART: histogram2d using compressed() arrays
    # -----------------------------
    freq, height_edges, field_edges = np.histogram2d(
        altitude_data.compressed(),
        field_data.compressed(),
        bins=[altitude_bins, field_bins],
    )

    # =====================================================
    # RETURN RAW FREQUENCY IF normalize=False
    # =====================================================
    if not normalize:
        return freq, height_edges, field_edges

    # =====================================================
    # NORMALIZATION (PyART)
    # =====================================================
    freq_sum = np.sum(freq, axis=1)
    point_thres = min_frac_thres * np.max(freq_sum)

    freq_sum_rep = np.repeat(freq_sum[:, None], freq.shape[1], axis=1)

    freq_norm = freq / freq_sum_rep
    freq_norm = np.ma.masked_where(freq_sum_rep < point_thres, freq_norm)

    return freq_norm, height_edges, field_edges

In [ ]:
zGrid_f, zGrid_c = ModelData_NSSL.GetZGrids() #*TESTING
[zTarget_f, zTarget_c] = ModelData_NSSL.GetZTarget(zGrid_f, zGrid_c) #*TESTING

def MakeCFADCalculation(radar3d, zlevels, levels_per_bin=4, normalize=True):
    # ------------------------------
    # 1. DEFINE BINS
    # ------------------------------
    field_bins = np.arange(0, 51, 1/levels_per_bin)

    zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])
    altitude_bins = zlevels
    # altitude_bins = zTarget_f #*TESTING

    altitude_levels = zlevels_center
    # altitude_levels = zTarget_c #TESTING
    

    # ------------------------------
    # 2. COMPUTE CFADs FOR BOTH SCHEMES
    # ------------------------------
    cfad, z_edges, dbz_edges = create_cfad_gridded(
        radar_gridded  = radar3d.values,
        altitude_levels = altitude_levels,
        field_bins      = field_bins,
        altitude_bins   = altitude_bins,
        min_frac_thres  = 0.1,
        normalize=normalize
    )

    # ------------------------------
    # 3. BIN CENTERS FOR PLOTTING
    # ------------------------------
    z_centers   = 0.5 * (z_edges[:-1] + z_edges[1:])
    dbz_centers = 0.5 * (dbz_edges[:-1] + dbz_edges[1:])

    return cfad, z_centers, dbz_centers

def NormalizeCFAD(freq, min_frac_thres=0.1):
    # =====================================================
    # NORMALIZATION (PyART)
    # =====================================================
    freq_sum = np.sum(freq, axis=1)
    point_thres = min_frac_thres * np.max(freq_sum)

    freq_sum_rep = np.repeat(freq_sum[:, None], freq.shape[1], axis=1)

    freq_norm = freq / freq_sum_rep
    # freq_norm = np.ma.masked_where(freq_sum_rep < point_thres, freq_norm)
    freq_norm = np.ma.array(freq_norm, mask=freq_sum_rep < point_thres) #correction for freq_norm

    return freq_norm

In [ ]:
####################################
#DATA LOADING
running = True #keep true when using job array
# running = False

In [ ]:
# #*THRESHOLD_TESTING
# def LoadSNR_constant(regionName): 

#     codeType = os.path.join("DataAnalysis", "Observation_Data")
#     dataType = "Radar_STNRatio_Constant"
#     outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
    
#     fileName = f"SNR_constant_{regionName}.nc"
#     filePath = os.path.join(outputDirectory, fileName)
    
#     SNR_constant = xr.open_dataarray(filePath)
#     print(f"Loaded From: {filePath}")
#     return SNR_constant

# SNR_constant = LoadSNR_constant(regionName=ModelData_NSSL.region)

In [ ]:
def GetFileNamePath(ModelData, t):
    
    # Build file name
    fileName = (
        f"ComparingCFADs_{ModelData.region}_"
        f"{ModelData.case}_spinup{ModelData.spinup_hours}hrs"
        f"_{ModelData.timeStrings[t]}"
        f".pkl"
    )
    
    # Build directory for radar timeseries
    outputDir = os.path.join(
        DirectoryManager.GetOutputDirectory(codeType, dataType),
        "ComparingCFADs",
        f"{ModelData.region}_{ModelData.case}_spinup{ModelData.spinup_hours}hrs",
    )

    os.makedirs(outputDir, exist_ok=True)
    
    # Full path to the .pkl file
    fileNamePath = os.path.join(outputDir, fileName)
    return fileNamePath

In [ ]:
def RunCalculations(ModelData_NSSL,ModelData_TEMPO,DirectoryManager,
                    zTarget=None):
    """
    Load CFADs from pickle file, or compute, save, and return them.
    """

    # ----------------------------------------------------------
    # 1. SETUP
    # ----------------------------------------------------------
    # Load radar mask once
    mask = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_NSSL)

    # Loading zlevels
    z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
    zlevels = np.loadtxt(z_levels_filePath)/1e3
    zGrid_f, zGrid_c = ModelData_NSSL.GetZGrids() #*TESTING

    # ----------------------------------------------------------
    # 2. LOOP OVER TIME
    # ----------------------------------------------------------
    for t in tqdm(loop_elements, desc="Computing CFADs"):

        # ------------------------------
        # GET FILENAMEPATH
        # ------------------------------        
        fileNamePath = GetFileNamePath(ModelData=ModelData_NSSL, t=t)
        
        # ------------------------------
        # LOAD DATA
        # ------------------------------
        radarNSSL  = ModelData_NSSL.GetDataTimestep_diag(t=t)["refl10cm"]
        radarTEMPO = ModelData_TEMPO.GetDataTimestep_diag(t=t)["refl10cm"]

        #Interpolating Z levels #*TESTING
        ################################
        radarNSSL = ModelData_NSSL.InterpolateVertical(radarNSSL,zGrid_f,zGrid_c,zTarget_f,zTarget_c)
        radarTEMPO = ModelData_TEMPO.InterpolateVertical(radarTEMPO,zGrid_f,zGrid_c,zTarget_f,zTarget_c)
        #################################

        radarNSSL = radarNSSL.where(lambda x: x > 0)
        radarTEMPO = radarTEMPO.where(lambda x: x > 0)
        # radarNSSL = radarNSSL.where(lambda x: x >= SNR_constant) #*THRESHOLD_TESTING
        # radarTEMPO = radarTEMPO.where(lambda x: x >= SNR_constant) #*THRESHOLD_TESTING
        
        radarNSSL  = radarNSSL.where(mask)
        radarTEMPO = radarTEMPO.where(mask)

        # ------------------------------
        # RAW HISTOGRAM (not normalized)
        # ------------------------------
        raw_NSSL,  z_centers, dbz_centers = MakeCFADCalculation(radarNSSL,  zlevels, normalize=False)
        raw_TEMPO, _, _               = MakeCFADCalculation(radarTEMPO, zlevels, normalize=False)

        # ----------------------------------------------------------
        # COMBINING INTO DICTIONARY
        # ----------------------------------------------------------
        results = {
            # --- RAW (unnormalized) CFADs ---
            "CFAD_NSSL_raw":  raw_NSSL,
            "CFAD_TEMPO_raw": raw_TEMPO,
        
            # --- AXES ---
            "z_centers": z_centers,
            "dbz_centers": dbz_centers,
        }
    
        # ----------------------------------------------------------
        # SAVE TO PKL
        # ----------------------------------------------------------
        with open(fileNamePath, "wb") as f:
            pickle.dump(results, f)
    
        print(f"Saved CFADs to: {fileNamePath}")

    return results

In [ ]:
#Loading for NSSL and TEMPO
if running:
    CFAD_results = RunCalculations(ModelData_NSSL,ModelData_TEMPO,DirectoryManager)

In [ ]:
####################################
#RECOMBINING
recombining = False #keep false when job_array is running
# recombining = True

In [ ]:
def Recombine():
    """
    Recombine per-timestep raw CFAD pickle files into total raw CFADs.
    """

    total_CFAD_NSSL  = None
    total_CFAD_TEMPO = None

    z_centers  = None
    dbz_centers = None

    for t in tqdm(range(ModelData_NSSL.Ntime), desc="Recombining CFADs"):

        fileNamePath = GetFileNamePath(ModelData=ModelData_NSSL, t=t)

        # ---- skip missing timesteps (important for job arrays)
        if not os.path.exists(fileNamePath):
            continue

        with open(fileNamePath, "rb") as f:
            results = pickle.load(f)

        raw_NSSL  = results["CFAD_NSSL_raw"]
        raw_TEMPO = results["CFAD_TEMPO_raw"]

        # store axes once
        if z_centers is None:
            z_centers  = results["z_centers"]
            dbz_centers = results["dbz_centers"]

        if total_CFAD_NSSL is None:
            total_CFAD_NSSL  = raw_NSSL.copy()
            total_CFAD_TEMPO = raw_TEMPO.copy()
        else:
            total_CFAD_NSSL  += raw_NSSL
            total_CFAD_TEMPO += raw_TEMPO

    # ----------------------------------------------------------
    # 2. NORMALIZE
    # ----------------------------------------------------------
    CFAD_NSSL  = NormalizeCFAD(total_CFAD_NSSL)
    CFAD_TEMPO = NormalizeCFAD(total_CFAD_TEMPO)
    
    # ----------------------------------------------------------
    # 3. NORMALIZE RAW OUTPUT
    # ----------------------------------------------------------
    total_CFAD_NSSL = np.ma.masked_where(total_CFAD_NSSL == 0, total_CFAD_NSSL)
    total_CFAD_TEMPO = np.ma.masked_where(total_CFAD_TEMPO == 0, total_CFAD_TEMPO)
            
    CFAD_results = {
        "CFAD_NSSL_raw":  total_CFAD_NSSL,
        "CFAD_TEMPO_raw": total_CFAD_TEMPO,
        "CFAD_NSSL":  CFAD_NSSL,
        "CFAD_TEMPO": CFAD_TEMPO,
        "z_centers": z_centers,
        "dbz_centers": dbz_centers,
    }
    return CFAD_results

In [ ]:
if recombining:
    CFAD_results = Recombine()

In [ ]:
####################################
#PLOTTING FUNCTIONS
plotting = False #keep false when running job array
# plotting = True

In [ ]:
#Loading for MRMS
def GetFileNamePath_MRMS(ModelData):
    
    # Build file name
    fileName = (
        f"MRMS_CFAD_{ModelData.region}_"
        f"{ModelData.case}_spinup{ModelData.spinup_hours}hrs.pkl"
    )
    
    # Build directory for radar timeseries
    outputDir = os.path.join(
        DirectoryManager.GetOutputDirectory(codeType, dataType),
        "ComparingCFADs"
    )

    os.makedirs(outputDir, exist_ok=True)
    
    # Full path to the .pkl file
    fileNamePath = os.path.join(outputDir, fileName)
    return fileNamePath

def LoadCFAD_MRMS(ModelData):
    fileNamePath = GetFileNamePath_MRMS(ModelData=ModelData)
    print(f"Loading precomputed CFADs from: {fileNamePath}")
    with open(fileNamePath, "rb") as f:
        return pickle.load(f)

if plotting:
    CFAD_results_MRMS=LoadCFAD_MRMS(ModelData_NSSL)

In [ ]:
def GetCFADColormap():

    # ---------------------------------------
    # 1. Define the LOG bin boundaries
    #    (you can adjust these if needed)
    # ---------------------------------------
    log_bins = np.array([1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2])

    # ---------------------------------------
    # 2. Define colors for each log interval
    #    (10 colors per log bin)
    # ---------------------------------------
    n_per_group = 10   # number of colors per log interval

    # These are the target group colors:
    grey     = np.array([0.60, 0.60, 0.60])
    green    = np.array([0.20, 0.70, 0.20])
    blue     = np.array([0.20, 0.40, 0.90])
    yellow   = np.array([0.95, 0.85, 0.20])
    orange   = np.array([1.00, 0.50, 0.05])
    red      = np.array([0.90, 0.10, 0.10])
    darkred  = np.array([0.60, 0.00, 0.00])

    # Ordered from low → high intensity
    group_colors = [grey, green, blue, yellow, orange, red, darkred]

    # ---------------------------------------
    # 3. Interpolate each group into 10 steps
    # ---------------------------------------
    colors_per_bin = []
    for i in range(len(group_colors)-1):
        start = group_colors[i]
        end   = group_colors[i+1]

        # generate gradient between two colors
        grad = np.linspace(start, end, n_per_group)
        colors_per_bin.append([tuple(c) for c in grad])

    # Flatten color list
    all_colors = [c for group in colors_per_bin for c in group]

    # ---------------------------------------
    # 4. Expand the log bins into sub-bins
    #    (10 subdivisions per decade)
    # ---------------------------------------
    expanded_bins = []
    for low, high in zip(log_bins[:-1], log_bins[1:]):
        sub = np.logspace(np.log10(low), np.log10(high), n_per_group + 1)
        expanded_bins.extend(sub[:-1])
    expanded_bins.append(log_bins[-1])
    expanded_bins = np.array(expanded_bins)

    # ---------------------------------------
    # 5. Build the colormap + norm
    # ---------------------------------------
    cmap = mcolors.ListedColormap(all_colors)
    norm = mcolors.BoundaryNorm(expanded_bins, len(all_colors))

    return cmap, norm

def GetCFADDiffColormap():

    # ---------------------------------------
    # 1. Define symmetric difference bins
    # ---------------------------------------
    diff_edges = np.array([-100, -10, -1, -0.1, 0, 0.1, 1, 10, 100])

    # ---------------------------------------
    # 2. Same colors as main CFAD colormap
    # ---------------------------------------
    n_per_group = 10

    grey     = np.array([0.60, 0.60, 0.60])
    green    = np.array([0.20, 0.70, 0.20])
    blue     = np.array([0.20, 0.40, 0.90])
    yellow   = np.array([0.95, 0.85, 0.20])
    orange   = np.array([1.00, 0.50, 0.05])
    red      = np.array([0.90, 0.10, 0.10])
    darkred  = np.array([0.60, 0.00, 0.00])

    group_colors = [grey, green, blue, yellow, orange, red, darkred]

    # ---------------------------------------
    # 3. Build the same 60-color list
    # ---------------------------------------
    colors_per_bin = []
    for i in range(len(group_colors)-1):
        grad = np.linspace(group_colors[i], group_colors[i+1], n_per_group)
        colors_per_bin.append([tuple(c) for c in grad])

    all_colors = [c for group in colors_per_bin for c in group]  # 60 colors

    # ---------------------------------------
    # 4. Build expanded bins matching 60 colors
    #    → evenly interpolate the 8 intervals to produce 60 bins
    # ---------------------------------------

    # We need exactly 60 boundaries for BoundaryNorm
    Ncolors = len(all_colors)
    Nedges = Ncolors + 1  # = 61

    # Interpolate across the full difference range
    expanded_bins = np.interp(
        np.linspace(0, len(diff_edges)-1, Nedges),
        np.arange(len(diff_edges)),
        diff_edges
    )

    # ---------------------------------------
    # 5. Build colormap + boundary norm
    # ---------------------------------------
    cmap = mcolors.ListedColormap(all_colors)
    norm = mcolors.BoundaryNorm(expanded_bins, Ncolors)

    return cmap, norm


In [ ]:
# def PlotCFAD_V1(CFAD_results, 
#              cmap='turbo', norm=None, colorbar_pad=0.01,
#              diff_cmap=None, diff_norm=None):
#     """
#     Plot CFAD for NSSL, TEMPO, and their difference using:
#       - cmap/norm for the main CFADs
#       - diff_cmap / diff_norm for the difference panel
#     """

#     # ------------------------------
#     # 1. LOAD CFAD RESULTS
#     # ------------------------------
#     cfad_NSSL  = CFAD_results["CFAD_NSSL"] * 1e2
#     cfad_TEMPO = CFAD_results["CFAD_TEMPO"] * 1e2
#     z_centers   = CFAD_results["z_centers"]
#     dbz_centers = CFAD_results["dbz_centers"]

#     # Difference (TEMPO - NSSL)
#     cfad_DIFF = cfad_NSSL - cfad_TEMPO

#     # ------------------------------
#     # 2. FIGURE LAYOUT
#     # ------------------------------
#     fig = plt.figure(figsize=(15, 8))
#     gs  = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], wspace=0.05)

#     ax1 = fig.add_subplot(gs[0, 0])
#     ax2 = fig.add_subplot(gs[0, 1], sharey=ax1)
#     ax3 = fig.add_subplot(gs[0, 2], sharey=ax1)

#     ax2.tick_params(left=False, labelleft=False)
#     ax3.tick_params(left=False, labelleft=False)

#     # ------------------------------
#     # 3. LEVELS FOR MAIN CFADS
#     # ------------------------------
#     if norm is not None:
#         levels = norm.boundaries
#     else:
#         levels = 100

#     # ------------------------------
#     # 4. PLOT NSSL CFAD
#     # ------------------------------
#     cf1 = ax1.contourf(
#         dbz_centers, z_centers, cfad_NSSL,
#         cmap=cmap, norm=norm,
#         levels=levels, extend='both'
#     )
#     ax1.set_title("NSSL")
#     ax1.set_xlabel("Reflectivity (dBZ)")
#     ax1.set_ylabel("Altitude (km)")

#     # ------------------------------
#     # 5. PLOT TEMPO CFAD
#     # ------------------------------
#     cf2 = ax2.contourf(
#         dbz_centers, z_centers, cfad_TEMPO,
#         cmap=cmap, norm=norm,
#         levels=levels, extend='both'
#     )
#     ax2.set_title("TEMPO")
#     ax2.set_xlabel("Reflectivity (dBZ)")

#     # ------------------------------
#     # 6. DIFFERENCE CFAD
#     # ------------------------------
#     cf3 = ax3.contourf(
#         dbz_centers, z_centers, cfad_DIFF,
#         cmap=diff_cmap,
#         norm=diff_norm,
#         levels=diff_norm.boundaries,
#         extend='both'
#     )
#     ax3.set_title("NSSL − TEMPO")
#     ax3.set_xlabel("Reflectivity (dBZ)")

#     # ------------------------------
#     # 7. TRIM EMPTY ALTITUDE ROWS
#     # ------------------------------
#     empty_NSSL  = np.all(cfad_NSSL.mask, axis=1)
#     empty_TEMPO = np.all(cfad_TEMPO.mask, axis=1)
#     empty_combined = empty_NSSL & empty_TEMPO

#     if empty_combined.any():
#         z_top = z_centers[np.where(empty_combined)[0][0]]
#     else:
#         z_top = z_centers.max()

#     for ax in (ax1, ax2, ax3):
#         ax.set_ylim(0, z_top)

#     # ------------------------------
#     # 8. MAIN COLORBAR (NSSL/TEMPO)
#     # ------------------------------
#     if norm is None:
#         cbar = fig.colorbar(cf2, ax=[ax1, ax2], location='right', pad=colorbar_pad)
#     else:
#         cbar = fig.colorbar(
#             cf2,
#             ax=[ax1, ax2],
#             location='right',
#             boundaries=norm.boundaries,
#             extend='both',
#             pad=colorbar_pad
#         )
#         tick_vals = [1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2]
#         cbar.set_ticks(tick_vals)
#         cbar.set_ticklabels(['1e−3','1e−2','1e−1','1','10','100'])

#     cbar.set_label("Normalized Frequency (%)")

#     # ------------------------------
#     # 9. DIFFERENCE COLORBAR
#     # ------------------------------
#     cbar2 = fig.colorbar(
#         cf3,
#         ax=ax3,
#         location='right',
#         boundaries=diff_norm.boundaries,
#         extend='both',
#         pad=colorbar_pad + 0.01
#     )
#     cbar2.set_label("Difference (%)")
#     diff_ticks = [-100, -10, -1, -0.1, 0, 0.1, 1, 10, 100]
#     cbar2.set_ticks(diff_ticks)
#     cbar2.set_ticklabels([f"{t}" for t in diff_ticks])

    

#     return fig


In [ ]:
def PlotCFAD_V2(CFAD_results, 
                CFAD_results_MRMS,
                cmap='turbo', norm=None, colorbar_pad=0.01,
                diff_cmap=None, diff_norm=None,
                normalize=True, common_top=None):

    # ------------------------------
    # 1. LOAD CFAD RESULTS
    # ------------------------------
    if normalize == True:
        cfad_NSSL  = CFAD_results["CFAD_NSSL"] * 1e2
        cfad_TEMPO = CFAD_results["CFAD_TEMPO"] * 1e2
        cfad_MRMS  = CFAD_results_MRMS["CFAD_MRMS"] * 1e2
    else:
        cfad_NSSL  = CFAD_results["CFAD_NSSL_raw"]
        cfad_TEMPO = CFAD_results["CFAD_TEMPO_raw"]
        cfad_MRMS  = CFAD_results_MRMS["CFAD_MRMS_raw"]
        norm = None

    # Vertical grids
    z_centers       = CFAD_results["z_centers"]
    dbz_centers     = CFAD_results["dbz_centers"]
    # z_centers       = CFAD_results["z_centers"]/1e3 #*TESTING
    # dbz_centers     = CFAD_results["dbz_centers"]/1e3 #*TESTING

    # MRMS vertical grid is in meters → convert to km
    z_centers_MRMS   = CFAD_results_MRMS["z_centers"] / 1e3
    dbz_centers_MRMS = CFAD_results_MRMS["dbz_centers"]

    # ------------------------------
    # 1B. DIFFERENCES
    # ------------------------------
    cfad_NSSL_interp = InterpolateCFAD_ModelToMRMS(cfad_NSSL,  z_centers, z_centers_MRMS)
    cfad_TEMPO_interp = InterpolateCFAD_ModelToMRMS(cfad_TEMPO, z_centers, z_centers_MRMS)
    diff_MRMS_NSSL  = cfad_MRMS - cfad_NSSL_interp
    diff_MRMS_TEMPO = cfad_MRMS - cfad_TEMPO_interp
    diff_NSSL_TEMPO = cfad_NSSL - cfad_TEMPO

    #Getting first All Nan Levels
    firstNan_1  = FirstNanLevel(diff_MRMS_NSSL)
    firstNan_2 = FirstNanLevel(diff_MRMS_TEMPO)
    firstNan_3    = FirstNanLevel(diff_NSSL_TEMPO)
    validLevels = [v for v in [firstNan_1, firstNan_2, firstNan_3] if v is not None]
    lowestNanLevel = min(validLevels) if validLevels else None


    # ------------------------------
    # 2. FIGURE AND AXES
    # ------------------------------
    fig = plt.figure(figsize=(20, 16))
    gs  = gridspec.GridSpec(2, 3, height_ratios=[1, 1], wspace=0.15, hspace=0.25)

    ax1 = fig.add_subplot(gs[0, 0])   # NSSL
    ax2 = fig.add_subplot(gs[0, 1])   # MRMS
    ax3 = fig.add_subplot(gs[0, 2])   # TEMPO

    ax4 = fig.add_subplot(gs[1, 0])   # empty / future
    ax5 = fig.add_subplot(gs[1, 1])   # empty / future
    ax6 = fig.add_subplot(gs[1, 2])   # NSSL - TEMPO Difference

    # ------------------------------
    # 3. LEVELS
    # ------------------------------
    if norm is not None:
        levels = norm.boundaries
    else:
        levels = 20

    # ------------------------------
    # 4. PLOT: NSSL
    # ------------------------------
    cf1 = ax1.contourf(
        dbz_centers, z_centers, cfad_NSSL,
        cmap=cmap, norm=norm,
        levels=levels, extend='both'
    )
    ax1.set_title("NSSL")
    ax1.set_xlabel("Reflectivity (dBZ)")
    ax1.set_ylabel("Altitude (km)")

    # ------------------------------
    # 5. PLOT: MRMS
    # ------------------------------
    cf2 = ax2.contourf(
        dbz_centers_MRMS, z_centers_MRMS, cfad_MRMS,
        cmap=cmap, norm=norm,
        levels=levels, extend='both'
    )
    ax2.set_title("MRMS")
    ax2.set_xlabel("Reflectivity (dBZ)")
    ax2.set_ylabel("Altitude (km)")

    # ------------------------------
    # 6. PLOT: TEMPO
    # ------------------------------
    cf3 = ax3.contourf(
        dbz_centers, z_centers, cfad_TEMPO,
        cmap=cmap, norm=norm,
        levels=levels, extend='both'
    )
    ax3.set_title("TEMPO")
    ax3.set_xlabel("Reflectivity (dBZ)")
    ax3.set_ylabel("Altitude (km)")

    # ------------------------------
    # DIFFERENCE COLORBAR
    # ------------------------------
    diffCmap = "bwr"
    if normalize == True:
        symmetric_minmax=5 
    else:
        # raw-count differences → auto-scale
        max_diff = np.nanmax([
            np.nanmax(np.abs(diff_MRMS_NSSL)),
            np.nanmax(np.abs(diff_MRMS_TEMPO)),
            np.nanmax(np.abs(diff_NSSL_TEMPO)),
        ])
        symmetric_minmax = max_diff
    vmin=-symmetric_minmax;vmax=symmetric_minmax
    diffNorm = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    diffLevels = np.linspace(vmin, vmax, 41)
    # ------------------------------
    # 7. DIFFERENCE CFAD (NSSL - MRMS)
    # ------------------------------
    cf4 = ax4.contourf(
        dbz_centers_MRMS, z_centers_MRMS, diff_MRMS_NSSL,
        # cmap=diff_cmap,
        cmap=diffCmap,
        # norm=diff_norm,
        norm=diffNorm,
        # levels=diff_norm.boundaries,
        levels=diffLevels,
        extend='both'
    )
    ax4.set_title("MRMS - NSSL")
    ax4.set_xlabel("Reflectivity (dBZ)")
    ax4.set_ylabel("Altitude (km)")

    # ------------------------------
    # 8. DIFFERENCE CFAD (NSSL - TEMPO)
    # ------------------------------
    cf5 = ax5.contourf(
        dbz_centers, z_centers, diff_NSSL_TEMPO,
        # cmap=diff_cmap,
        cmap=diffCmap,
        # norm=diff_norm,
        norm=diffNorm,
        # levels=diff_norm.boundaries,
        levels=diffLevels,
        extend='both'
    )
    ax5.set_title("NSSL − TEMPO")
    ax5.set_xlabel("Reflectivity (dBZ)")
    ax5.set_ylabel("Altitude (km)")

    # ------------------------------
    # 9. DIFFERENCE CFAD (TEMPO - MRMS)
    # ------------------------------
    cf6 = ax6.contourf(
        dbz_centers_MRMS, z_centers_MRMS, diff_MRMS_TEMPO,
        # cmap=diff_cmap,
        cmap=diffCmap,
        # norm=diff_norm,
        norm=diffNorm,
        # levels=diff_norm.boundaries,
        levels=diffLevels,
        extend='both'
    )
    ax6.set_title("MRMS − TEMPO")
    ax6.set_xlabel("Reflectivity (dBZ)")
    ax6.set_ylabel("Altitude (km)")
    


    # ------------------------------
    # 8. TRIM EMPTY LAYERS
    # ------------------------------
    empty_NSSL  = np.all(cfad_NSSL.mask, axis=1)
    empty_TEMPO = np.all(cfad_TEMPO.mask, axis=1)
    empty_combined = empty_NSSL & empty_TEMPO

    if empty_combined.any():
        z_top = z_centers[np.where(empty_combined)[0][0]]
    else:
        z_top = z_centers.max()

    #SETTING Y LIM
    # Unified altitude range for top row
    if common_top is None:
        common_top = z_centers_MRMS[lowestNanLevel-1]      # MPAS top height (km)
    axes = fig.get_axes()
    for ax in axes:#(ax1, ax2, ax3, ax4, ax5, ax6): 
        ax.set_ylim(z_centers_MRMS.min(), common_top)
    SetEvenTicks(axes, dim="y", n_ticks=6, decimals=1)

    # ==============================================
    # 9. MAIN COLORBAR – match height of top-row axes
    # ==============================================
    bb_top = ax1.get_position()
    
    cbar_ax1 = fig.add_axes([
        ax3.get_position().x1 + 0.01,  # left
        bb_top.y0,                     # bottom
        0.015,                         # width
        bb_top.height                  # height
    ])
    
    # If no norm: autoscale (0 → 100%)
    if norm is None:
        cbar = fig.colorbar(
            cf1,
            cax=cbar_ax1,
            extend='both'
        )
    else:
        cbar = fig.colorbar(
            cf1,
            cax=cbar_ax1,
            boundaries=norm.boundaries,
            extend='both'
        )
        tick_vals = [1e-3, 1e-2, 1e-1, 1, 10, 100]
        cbar.set_ticks(tick_vals)
        cbar.set_ticklabels(['1e−3','1e−2','1e−1','1','10','100'])

    if normalize==True:
        cbar.set_label("Normalized Frequency (%)")
    else:
        cbar.set_label("Count")
        
        
    # ==================================================
    # 10. DIFFERENCE COLORBAR – match height of bottom row
    # ==================================================
    bb_bot = ax4.get_position()
    
    cbar_ax2 = fig.add_axes([
        ax6.get_position().x1 + 0.01,   # right of bottom right panel
        bb_bot.y0,                      # bottom aligned with row
        0.015,                          # width
        bb_bot.height                   # height = row height
    ])
    
    # one colorbar for ALL difference plots
    cbar2 = fig.colorbar(
        cf4,               # any diff panel works, all use same cmap/norm/levels
        cax=cbar_ax2,
        cmap=diffCmap,
        norm=diffNorm,
        boundaries=diffLevels,
        extend='both'
    )
    
    cbar2.set_label("Difference (%)")

    # ==============================================
    # 11. SUPTITLE
    # ==============================================
    fig.suptitle(
        f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
        fontsize=20,
        y=0.93,
        fontweight="bold")

    return fig, common_top

def FirstNanLevel(array):
    levelAllNan = np.all(np.isnan(array), axis=1)
    idx = np.where(levelAllNan)[0]
    return idx[0] if idx.size > 0 else None

def InterpolateCFAD_ModelToMRMS(cfad_model, z_model, z_mrms):
    """
    Interpolate a model CFAD (NSSL or TEMPO) onto the MRMS vertical grid.
    """

    cfad_filled = np.ma.filled(cfad_model, np.nan)      # convert masked to NaN
    n_bins = cfad_model.shape[1]

    cfad_interp = np.zeros((len(z_mrms), n_bins))

    for b in range(n_bins):
        # vertical interpolation along altitude
        cfad_interp[:, b] = np.interp(
            z_mrms,
            z_model,
            cfad_filled[:, b],
            left=np.nan,
            right=np.nan
        )

    # rebuild masked array: mask where either nan
    cfad_interp = np.ma.masked_invalid(cfad_interp)

    return cfad_interp

In [ ]:
def GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory):
    outputSubDirectory = f"{ModelData_1.region}_{ModelData_1.case}_{ModelData_1.spinup_hours}hrs"
    
    outputFilePath = os.path.join(
        outputPlottingDirectory,
        "ComparingCFADs",
        outputSubDirectory)
    os.makedirs(outputFilePath, exist_ok=True)
    return outputFilePath

def SaveFigure(fig, ModelData_1,ModelData_2):
    """
    Saves a figure to corresponding directory.
    """
    # --- Define output subdirectory and file path ---
    outputFilePath = GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory)
    outputFile = os.path.join(outputFilePath,f"ComparingCFADs_{ModelData_1.mpType}vsMRMSvs{ModelData_2.mpType}.png")

    # --- Save figure ---
    fig.savefig(outputFile, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

def SaveFigure_NotNormalized(fig, ModelData_1,ModelData_2):
    """
    Saves a figure to corresponding directory.
    """
    # --- Define output subdirectory and file path ---
    outputFilePath = GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory)
    outputFile = os.path.join(outputFilePath,f"ComparingCFADs_NotNormalized_{ModelData_1.mpType}vsMRMSvs{ModelData_2.mpType}.png")

    # --- Save figure ---
    fig.savefig(outputFile, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
####################################
#PLOTTING

In [ ]:
if plotting:
    custom_cmap, custom_norm = GetCFADColormap()
    custom_cmap_diff, custom_norm_diff = GetCFADDiffColormap()
    
    fig, common_top = PlotCFAD_V2(
        CFAD_results,
        CFAD_results_MRMS,
        cmap=custom_cmap,
        norm=custom_norm,
        colorbar_pad=0.01,
        diff_cmap=custom_cmap_diff,
        diff_norm=custom_norm_diff
    )
     
    SaveFigure(fig, ModelData_NSSL, ModelData_TEMPO)

In [ ]:
if plotting:
    custom_cmap, custom_norm = GetCFADColormap()
    custom_cmap_diff, custom_norm_diff = GetCFADDiffColormap()
    
    fig, _ = PlotCFAD_V2(
        CFAD_results,
        CFAD_results_MRMS,
        cmap=custom_cmap,
        norm=None,
        colorbar_pad=0.01,
        diff_cmap=custom_cmap_diff,
        diff_norm=custom_norm_diff,
        normalize=False,
        common_top=common_top
    )
    
    SaveFigure_NotNormalized(fig, ModelData_NSSL, ModelData_TEMPO)

In [ ]:
####################################
#PLOTTING ALL SIMULATIONS
plotting = False #keep false when job array is running
# plotting = True

In [ ]:
def GetFigureFilePath(region,case,spinup_hours,
               extension="png"):

    inputFilePath = os.path.join(
        outputPlottingDirectory,
        "ComparingCFADs")

    fileName="ComparingCFADs_NSSLvsMRMSvsTEMPO" if region != "PRECIP" else "ComparingCFADs_NSSLvsPRECIPvsTEMPO"

    
    # --- Define output subdirectory ---
    inputSubDirectory = f"{region}_{case}_{spinup_hours}hrs"
    load_dir = os.path.join(inputFilePath, inputSubDirectory)
    # --- File path ---
    inputFilePath = os.path.join(
        load_dir,
        f"{fileName}.{extension}"
    )
    return inputFilePath

def GetFilePaths():
    caseList = ConsolidateFigures_CLASS.GetCaseList()
    filePaths = []
    for region, case, spinup_hours in caseList:
        filePaths.append(GetFigureFilePath(region,case,spinup_hours))
    return filePaths

In [ ]:
if plotting:
    sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
    from CLASSES_Plotting import ConsolidateFigures_CLASS

In [ ]:
if plotting:
    filePaths = GetFilePaths()
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(6, 6),
                                                     wspace=0.02,hspace=0.02,
                                                     dpi=900)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig,dpi=900, saveDirectory=os.path.join(outputPlottingDirectory,"ComparingCFADs"),fileName="ComparingCFADs")